In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import zipfile
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import io
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.nn as nn
import numpy as np
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
import json




In [ ]:

os.makedirs("/content/drive/MyDrive", exist_ok=True)

In [ ]:


print(os.path.exists("/content/drive/MyDrive/nih-chest-x-ray-14-224x224-resized.zip"))

True


In [ ]:


ZIP_PATH = "/content/drive/MyDrive/nih-chest-x-ray-14-224x224-resized.zip"

zip_file = zipfile.ZipFile(ZIP_PATH, 'r')

print("Total files in ZIP:", len(zip_file.namelist()))

Total files in ZIP: 112125


In [ ]:


with zip_file.open("Data_Entry_2017.csv") as f:
    labels_df = pd.read_csv(f)

labels_df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,058Y,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,058Y,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,058Y,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,081Y,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,081Y,F,PA,2582,2991,0.143,0.143,NaN


In [ ]:
ALL_LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema",
    "Fibrosis", "Pleural_Thickening", "Hernia",
    "No Finding"
]

print("Total labels:", len(ALL_LABELS))

Total labels: 15


In [ ]:
data = []

for _, row in labels_df.iterrows():

    image_name = row["Image Index"]
    labels = row["Finding Labels"].split("|")

    image_path = f"images-224/images-224/{image_name}"

    multi_label = [0] * len(ALL_LABELS)

    if labels == ["No Finding"]:
        idx = ALL_LABELS.index("No Finding")
        multi_label[idx] = 1
    else:
        for l in labels:
            if l in ALL_LABELS:
                idx = ALL_LABELS.index(l)
                multi_label[idx] = 1

    data.append({
        "image_name": image_path,
        "labels": multi_label
    })

dataset_df = pd.DataFrame(data)

print("Dataset size:", len(dataset_df))

Dataset size: 112120


In [ ]:


class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, zip_file, transform=None):
        self.df = dataframe
        self.zip_file = zip_file
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_name = row["image_name"]
        label = row["labels"]

        with self.zip_file.open(image_name) as f:
            image = Image.open(io.BytesIO(f.read())).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.float)

        return image, label

In [ ]:
sample = dataset_df.iloc[0]

print("Image path:", sample["image_name"])
print("Label vector:", sample["labels"])
print("Label length:", len(sample["labels"]))

Image path: images-224/images-224/00000001_000.png
Label vector: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Label length: 15


In [ ]:

train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 78484
Validation size: 16818
Test size: 16818


In [ ]:


train_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomCrop((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = ChestXrayDataset(
    train_df,
    zip_file,
    transform=train_transforms
)

val_dataset = ChestXrayDataset(
    val_df,
    zip_file,
    transform=val_transforms
)

test_dataset = ChestXrayDataset(
    test_df,
    zip_file,
    transform=val_transforms
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 78484
Validation dataset: 16818
Test dataset: 16818


In [ ]:


BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("✅ DataLoaders ready")

✅ DataLoaders ready


In [ ]:
images, labels = next(iter(train_loader))

print("Image shape:", images.shape)
print("Label shape:", labels.shape)

Image shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32, 15])


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# تحميل النموذج المسبق DenseNet121
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

# تعديل طبقة الخرج لتتناسب مع 15 فئة
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 15)

# نقل النموذج إلى الجهاز الحالي (CPU حالياً)
model = model.to(device)

print(model)

cuda
DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        

In [ ]:

labels_array = np.array(train_df["labels"].tolist())

pos_counts = labels_array.sum(axis=0)
neg_counts = labels_array.shape[0] - pos_counts

pos_weight = neg_counts / (pos_counts + 1e-5)

pos_weight = np.log1p(pos_weight)

pos_weight = torch.tensor(
    pos_weight,
    dtype=torch.float
).to(device)

In [ ]:

for param in model.features.parameters():
    param.requires_grad = False

for param in model.features.denseblock4.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

# Loss function (مهم جداً لمشكلة imbalance)
pos_weight_adj = pos_weight.clone()
no_finding_idx = ALL_LABELS.index("No Finding")

# تقليل تأثير No Finding
pos_weight_adj[no_finding_idx] = pos_weight_adj[no_finding_idx] * 0.3

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_adj)

# Optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-5,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=1
)

print("Loss, Optimizer and Scheduler ready ✅")

Loss, Optimizer and Scheduler ready ✅


In [ ]:
print(next(model.parameters()).device)
print(pos_weight.device)

cuda:0
cuda:0


In [ ]:


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    return avg_loss

In [ ]:


CHECKPOINT_PATH = "/content/drive/MyDrive/model_checkpoint.pth"

start_epoch = 0
best_auc = 0

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=False   # ✅ هذا هو الحل
    )

    model.load_state_dict(checkpoint["model_state"])

    start_epoch = checkpoint["epoch"] + 1
    best_auc = checkpoint.get("best_auc", 0)

    print(f"✅ Loaded checkpoint from epoch {start_epoch}")
else:
    print("⚠️ No checkpoint found, starting from scratch")

✅ Loaded checkpoint from epoch 24


In [ ]:
def validate(model, valloader, criterion, device):
    model.eval()

    val_loss = 0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for inputs, labels in valloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = torch.sigmoid(outputs)

            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)

    thresholds = get_optimal_thresholds(all_labels, all_preds)

    preds_bin = np.zeros_like(all_preds)

    for i in range(all_preds.shape[1]):
        preds_bin[:, i] = (all_preds[:, i] > thresholds[i]).astype(int)

    auc = roc_auc_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, preds_bin, average="macro")

    return {
        "loss": val_loss / len(valloader),
        "avg_auc": auc,
        "avg_f1": f1,
        "thresholds": thresholds
    }

In [ ]:


def get_optimal_thresholds(labels, preds):
    thresholds = []

    for i in range(preds.shape[1]):

        best_f1 = 0
        best_t = 0.5

        for t in np.arange(0.1, 0.9, 0.05):
            pred_bin = (preds[:, i] > t).astype(int)

            f1 = f1_score(
                labels[:, i],
                pred_bin,
                zero_division=0
            )

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds.append(best_t)

    return thresholds

In [ ]:
checkpoint = torch.load(
    "/content/drive/MyDrive/best_model.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state"])

print("✅ Model loaded successfully")

✅ Model loaded successfully


In [ ]:
ADDITIONAL_EPOCHS = 1
END_EPOCH = start_epoch + ADDITIONAL_EPOCHS

patience = 2
no_improve = 0
best_f1 = checkpoint.get("best_f1", 0)

# ================= HISTORY =================
history = {
    "train_loss": [],
    "val_loss": [],
    "auc": [],
    "f1_05": [],
    "f1_custom": []
}

# ================= TRAINING LOOP =================
for epoch in range(start_epoch, END_EPOCH):

    print(f"\nEpoch [{epoch+1}/{END_EPOCH}]")

    # ================= TRAIN =================
    model.train()

    train_loss = 0

    loop = tqdm(train_loader)

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)

    # ================= VALIDATION =================
    model.eval()

    val_loss = 0

    all_labels = []
    all_outputs = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            all_labels.append(labels.cpu().numpy())

            all_outputs.append(
                torch.sigmoid(outputs).cpu().numpy()
            )

    val_loss /= len(val_loader)

    all_labels = np.vstack(all_labels)

    all_outputs = np.vstack(all_outputs)

    # ================= AUC =================
    auc = roc_auc_score(
        all_labels,
        all_outputs,
        average="macro"
    )

    # ================= THRESHOLDS =================
    best_thresholds = get_optimal_thresholds(
        all_labels,
        all_outputs
    )

    # smoothing thresholds
    if epoch == start_epoch:

        global_thresholds = best_thresholds

    else:

        global_thresholds = [

            0.8 * old + 0.2 * new

            for old, new in zip(
                global_thresholds,
                best_thresholds
            )
        ]

    # ================= F1 (0.5) =================
    preds_05 = (
        all_outputs > 0.5
    ).astype(int)

    f1_05 = f1_score(
        all_labels,
        preds_05,
        average="macro"
    )

    # ================= F1 (custom thresholds) =================
    preds = np.zeros_like(all_outputs)

    for i in range(all_outputs.shape[1]):

        preds[:, i] = (
            all_outputs[:, i] > global_thresholds[i]
        ).astype(int)

    f1_custom = f1_score(
        all_labels,
        preds,
        average="macro"
    )

    # ================= PRINT RESULTS =================
    print(f"Train Loss: {train_loss:.4f}")

    print(f"Val Loss: {val_loss:.4f}")

    print(f"AUC: {auc:.4f}")

    print(f"F1 (0.5): {f1_05:.4f}")

    print(f"F1 (custom): {f1_custom:.4f}")

    # ================= SAVE HISTORY =================
    history["train_loss"].append(train_loss)

    history["val_loss"].append(val_loss)

    history["auc"].append(auc)

    history["f1_05"].append(f1_05)

    history["f1_custom"].append(f1_custom)

    # ================= SCHEDULER =================
    scheduler.step(1 - f1_custom)

    # ================= CHECKPOINT =================
    torch.save({

        "epoch": epoch,

        "model_state": model.state_dict(),

        "optimizer_state": optimizer.state_dict(),

        "best_f1": best_f1,

        "history": history

    }, CHECKPOINT_PATH)

    print("💾 Checkpoint saved")

    # ================= BEST MODEL =================
    if f1_custom > best_f1:

        best_f1 = f1_custom

        no_improve = 0

        torch.save({

            "model_state": model.state_dict(),

            "labels": ALL_LABELS,

            "thresholds": global_thresholds,

            "best_f1": best_f1,

            "auc": auc,

            "model_name": "DenseNet121",

            "input_size": 224,

            "best_epoch": epoch + 1,

            "history": history

        }, "/content/drive/MyDrive/best_model.pth")

        print("✅ Best model updated")

    else:

        no_improve += 1

        print(f"⚠️ No improvement ({no_improve}/{patience})")

    # ================= EARLY STOP =================
    if no_improve >= patience:

        print("🛑 Early stopping triggered")

        break

    print("-" * 40)


Epoch [25/25]


100%|██████████| 2453/2453 [10:45<00:00,  3.80it/s, loss=0.289]


Train Loss: 0.2135
Val Loss: 0.3228
AUC: 0.8193
F1 (0.5): 0.3155
F1 (custom): 0.3571
💾 Checkpoint saved
⚠️ No improvement (1/2)
----------------------------------------


In [ ]:
checkpoint = torch.load(
    "/content/drive/MyDrive/best_model.pth",
    map_location=device,
    weights_only=False
)

print(type(checkpoint))

print(
    checkpoint.keys()
    if isinstance(checkpoint, dict)
    else "NO KEYS - state_dict only"
)

<class 'dict'>
dict_keys(['model_state', 'labels', 'thresholds', 'best_f1', 'auc', 'model_name', 'input_size', 'best_epoch', 'history'])


In [ ]:
checkpoint = torch.load(
    "/content/drive/MyDrive/best_model.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state"])

ALL_LABELS = checkpoint["labels"]

FINAL_THRESHOLDS = checkpoint["thresholds"]

print("Best F1:", checkpoint["best_f1"])

print("AUC:", checkpoint["auc"])

print("Model:", checkpoint["model_name"])

print("Best Epoch:", checkpoint["best_epoch"])

model.eval()

print("✅ Best model loaded")

Best F1: 0.35971619608409094
AUC: 0.8214181236156025
Model: DenseNet121
Best Epoch: 24
✅ Best model loaded


In [ ]:
model.eval()

all_labels = []
all_outputs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        outputs = torch.sigmoid(outputs)

        all_labels.append(labels.cpu().numpy())
        all_outputs.append(outputs.cpu().numpy())

all_labels = np.vstack(all_labels)
all_outputs = np.vstack(all_outputs)

print("Labels shape:", all_labels.shape)
print("Outputs shape:", all_outputs.shape)

Labels shape: (16818, 15)
Outputs shape: (16818, 15)


In [ ]:

import json

FINAL_THRESHOLDS = checkpoint["thresholds"]

with open("/content/drive/MyDrive/final_thresholds.json", "w") as f:
    json.dump(FINAL_THRESHOLDS, f)

print("✅ Thresholds saved")

✅ Thresholds saved


In [ ]:
for label, t in zip(ALL_LABELS, FINAL_THRESHOLDS):
    print(f"{label:20s} : {t:.2f}")

In [ ]:
labels_path = "/content/drive/MyDrive/final_labels.json"
with open(labels_path, "w") as f:
    json.dump(ALL_LABELS, f)
print("✅ Labels saved")

final_info = {
    "Model": checkpoint["model_name"],
    "Classes": len(ALL_LABELS),
    "Input_Size": checkpoint["input_size"],
    "Best_F1": checkpoint["best_f1"],
    "Best_AUC": checkpoint["auc"],
    "Best_Epoch": checkpoint["best_epoch"],
    "Batch_Size": BATCH_SIZE,
    "Learning_Rate": optimizer.param_groups[0]["lr"],
    "Optimizer": "AdamW",
    "Loss_Function": "BCEWithLogitsLoss"
}


In [ ]:
with open(
    "/content/drive/MyDrive/model_info.json",
    "w"
) as f:
    json.dump(final_info, f, indent=4)
print("✅ Model info saved")


✅ Model info saved
